###### import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.io import fits
from scipy.optimize import curve_fit
from scipy.integrate import quad
import scipy.integrate as spi
import re
from astropy.wcs import WCS
from matplotlib.colors import LinearSegmentedColormap
from astropy.io import fits
from astropy.convolution import convolve, Gaussian2DKernel
import os
from astropy.nddata import Cutout2D
from astropy.visualization import ZScaleInterval
csv = pd.read_csv("../data/raw/catalogue/jwst_bubble_properties_A.txt")

hdu = fits.open("../data/raw/fits/jw02107-o039_t018_miri_f770w_i2d.fits")
data = hdu[1].data

header = hdu[0].header


pixel_area_arcsec = 0.0121 # Nominal pixel area in arcsec^2
pixel_to_arcsec = 0.0121**(1/2) # pixel in arcsec
galaxydistance = 9000 # kpc
# Physical Size (pc) = 9,000kpc×0.1arcsec=900pc
arcsec_to_pc = galaxydistance * pixel_to_arcsec / 206265

pc_to_pixel = 6 # (1/arcsec_to_pc) * (1/0.0121**(1/2))


# image_width_pixels = 2092
# image_height_pixels =5648

expected_x, expected_y = 1809, 335
bbx,bby= 1467, 219
difx,dify= expected_x -bbx, expected_y -bby
header = hdu[1].header

image_center_ra = header['CRVAL1']
image_center_dec = header['CRVAL2']
image_width_pixels = header['NAXIS1']
image_height_pixels = header['NAXIS2']


CDELT1_deg_per_pixel = header['CDELT1']
plate_scale = abs(CDELT1_deg_per_pixel)
CRPIX1 = 2791.4534894068606
CRPIX2 = 1056.864371503309
CRVAL1 = 24.175301542591694 # RA reference in decimal degrees
CRVAL2 = 15.736458360554542 # Dec reference in decimal degrees
CDELT1 = 3.08130462259148E-05 # Degrees per pixel (RA direction)
CDELT2 = 3.08130462259148E-05 # Degrees per pixel (Dec direction)
PC1_1 = 0.3421816484245103
PC1_2 = -0.9396338220187079
PC2_1 = -0.9396338220187079
PC2_2 = -0.3421816484245103

def ra_dec_to_deg(ra_str, dec_str):
    ra_pattern = re.compile(r'(\d+)d(\d+)m(\d+\.\d+)s')
    dec_pattern = re.compile(r'(\d+)d(\d+)m(\d+\.\d+)s')

    ra_deg, ra_min, ra_sec = map(float, ra_pattern.match(ra_str).groups())
    dec_deg, dec_min, dec_sec = map(float, dec_pattern.match(dec_str).groups())
    
    ra_decimal = ra_deg + ra_min / 60 + ra_sec / 3600
    dec_decimal = dec_deg + dec_min / 60 + dec_sec / 3600
    # print(ra_decimal)
    return ra_decimal, dec_decimal

def ra_dec_to_pixel(ra_decimal, dec_decimal, CRPIX1, CRPIX2, CRVAL1, CRVAL2, CDELT1, CDELT2, PC1_1, PC1_2, PC2_1, PC2_2):
    """
    Converts RA and Dec in decimal degrees to pixel coordinates.
    
    Parameters:
    - ra_decimal: RA in decimal degrees
    - dec_decimal: Dec in decimal degrees
    - CRPIX1, CRPIX2: Reference pixel coordinates for RA and Dec (from header)
    - CRVAL1, CRVAL2: Reference RA and Dec (from header)
    - CDELT1, CDELT2: Pixel scale in RA and Dec directions (degrees per pixel)
    - PC1_1, PC1_2, PC2_1, PC2_2: Linear transformation matrix elements (from header)
    
    Returns:
    - (x_pixel, y_pixel): Pixel coordinates
    """
    delta_RA = ra_decimal - CRVAL1
    delta_Dec = dec_decimal - CRVAL2
    
    delta_X = PC1_1 * delta_RA + PC1_2 * delta_Dec
    delta_Y = PC2_1 * delta_RA + PC2_2 * delta_Dec
    
    x_pixel = CRPIX1 + delta_X / CDELT1
    y_pixel = CRPIX2 + delta_Y / CDELT2
    
    return x_pixel, y_pixel




angle = 2.78 /(9.84*10**3)

print("angle from center",angle*180/np.pi)
x = np.zeros(len(csv), dtype=float)
y = np.zeros(len(csv), dtype=float)

path_for_bubbles = '../verify/bubbles'
path_for_random = '../verify/random'

semi_maj_pixels = np.zeros(len(csv), dtype=float)

#10,24d08m59.3s,15d48m6.1s
print("i am finding hte biggest bubble", ra_dec_to_deg("24d08m59.3s", "15d48m6.1s"))
degrees = csv['PA_DEG']
coord= open("../verify/coord.txt", 'w')
for i, row in csv.iterrows():

    ra_decimal, dec_decimal = ra_dec_to_deg(row['RA_DMS'], row['DEC_DMS'])
    semi_maj_pixels[i] = row['SEMI_MAJ_PC'] /6.126
    x_pixel, y_pixel = ra_dec_to_pixel(ra_decimal, dec_decimal, CRPIX1, CRPIX2, CRVAL1, CRVAL2, CDELT1, CDELT2, PC1_1, PC1_2, PC2_1, PC2_2)
    
    x[i] = x_pixel #+difx
    y[i] = y_pixel #+dify

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(data[0:1300, 0:2500], cmap='gray', origin='lower', vmin=9.5, vmax=16) #vmin and vmax are part of the resolution. and also i think the command is dpi

#1506,24d11m9.6s,15d47m1.0s,582,493,552,70,4,89,2.78

for i, (x_i, y_i) in enumerate(zip(x, y)):

    r_i = semi_maj_pixels[i-1] # Semi-major axis in pixels ##########################################################################################
    coord.write(f"{x_i} {y_i} {r_i}\n") # Save values as space-separated
    
    
    circle = plt.Circle((x_i, y_i), r_i, color='red', fill=False, linewidth=1)
    ax.add_patch(circle)
    
    pa_360 = degrees[i]
    north_angle_rad = np.deg2rad(70.5 + 180) # Rotation adjustment
    


coord.close()


ax.scatter(x, y, color='blue', marker='x', s=13, label="Bubble Centers")

ax.set_title('NGC628 with Bubbles Overlay')
ax.set_xlabel('X Axis')
ax.set_ylabel('Y Axis')
ax.axis('off')

save_path_png = '../outputs/figures/all_bubbles_overlay.png'
plt.savefig(save_path_png, dpi=400, bbox_inches='tight', pad_inches=0)
plt.show()




degrees = csv['PA_DEG']






print("All bubbles have been extracted and saved.")

